In [235]:
import re
import string
import numpy as np
import pandas as pd
from collections import Counter
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

pd.set_option('display.max_colwidth', None)

In [236]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 14245363468419526618
xla_global_id: -1
]


## Load Datasets

In [237]:
train = pd.read_parquet('/home/inductive_anks/Machine Learning/Competitions/Zindi-Africa/Machine-Translation/data/raw/train-00000-of-00001.parquet')

In [238]:
test = pd.read_parquet('/home/inductive_anks/Machine Learning/Competitions/Zindi-Africa/Machine-Translation/data/raw/test-00000-of-00001.parquet')

In [239]:
validate = pd.read_parquet('/home/inductive_anks/Machine Learning/Competitions/Zindi-Africa/Machine-Translation/data/raw/validation-00000-of-00001.parquet')

In [240]:
train.head()

,ID,translation
0,ID_18897661270129,"{'dyu': 'A bi ji min na', 'fr': 'Il boit de l’eau.'}"
1,ID_18479132727846,"{'dyu': 'A le dalakolontɛ lon bɛ.', 'fr': 'Il se plaint toujours.'}"
2,ID_18164131280307,"{'dyu': 'Mun? Fɛn dɔ.', 'fr': 'Quoi ? Quelque chose.'}"
3,ID_18344573728152,"{'dyu': 'O bɛ bi bɔra fo Gubeta.', 'fr': 'Tous sortent excepté Gubetta.'}"
4,ID_18127342282717,"{'dyu': 'A ale lo bi da bugɔ la!', 'fr': 'Ah ! c’est lui… il sonne…'}"


In [241]:
test.head()

,ID,translation
0,ID_17345911362699,"{'dyu': 'An kelen duron le tun be yi', 'fr': '0'}"
1,ID_173626847.3381,"{'dyu': 'O ka papiye farana.', 'fr': '0'}"
2,ID_17704632382547,"{'dyu': 'N tɔrɔla kɔ tuguni', 'fr': '0'}"
3,ID_19793499384156,"{'dyu': 'I tun b'a daminɛ tan kɛ.', 'fr': '0'}"
4,ID_17802727385575,"{'dyu': 'A kɛra ka ban.', 'fr': '0'}"


In [242]:
validate.head()

,ID,translation
0,ID_17914990255818,"{'dyu': 'I tɔgɔ bi cogodɔ', 'fr': 'Tu portes un nom de fantaisie.'}"
1,ID_18135961264225,"{'dyu': 'Puɛn saba fɔlɔ.', 'fr': 'Trois points d’avance.'}"
2,ID_18161475265686,"{'dyu': 'Tile bena', 'fr': 'Le soleil s’est couché.'}"
3,ID_17305345266745,"{'dyu': 'cogoya kelen', 'fr': 'Mêmes mouvements.'}"
4,ID_18106593267767,"{'dyu': 'N ma daraka dun ban.', 'fr': 'Je n’ai pas encore déjeuné.'}"


In [243]:
print('Length of train data: ', len(train))
print('Length of test data: ', len(test))
print('Length of validate data: ', len(validate))

Length of train data:  8065
Length of test data:  1393
Length of validate data:  1471


## Finding Punctuations in the Data

In [244]:
train['dyu'] = train['translation'].apply(lambda x: x['dyu'])
train['fr'] = train['translation'].apply(lambda x: x['fr'])

test['dyu'] = test['translation'].apply(lambda x: x['dyu'])

validate['dyu'] = validate['translation'].apply(lambda x: x['dyu'])
validate['fr'] = validate['translation'].apply(lambda x: x['fr'])


In [245]:
def find_punctuation(text):
    punctuation = re.findall(r'[^\w\s]', text)
    return punctuation

In [246]:
train['dyu_punctuation'] = train['dyu'].apply(find_punctuation)
train['fr_punctuation'] = train['fr'].apply(find_punctuation)

validate['dyu_punctuation'] = validate['dyu'].apply(find_punctuation)
validate['fr_punctuation'] = validate['fr'].apply(find_punctuation)

test['dyu_punctuation'] = test['dyu'].apply(find_punctuation)

train_dyu_punctuation_count = Counter([p for sublist in train['dyu_punctuation'] for p in sublist])
train_fr_punctuation_count = Counter([p for sublist in train['fr_punctuation'] for p in sublist])

validate_dyu_punctuation_count = Counter([p for sublist in validate['dyu_punctuation'] for p in sublist])
validate_fr_punctuation_count = Counter([p for sublist in validate['fr_punctuation'] for p in sublist])

test_dyu_punctuation_count = Counter([p for sublist in test['dyu_punctuation'] for p in sublist])


In [247]:
print("Train Dyula Punctuation Count:", train_dyu_punctuation_count)
print("\nTrain French Punctuation Count:", train_fr_punctuation_count)

print("\nValidate Dyula Punctuation Count:", validate_dyu_punctuation_count)
print("\nValidate French Punctuation Count:", validate_fr_punctuation_count)

print("\nTest Dyula Punctuation Count:", test_dyu_punctuation_count)

Train Dyula Punctuation Count: Counter({'.': 2247, "'": 1037, '?': 540, ',': 210, '!': 63, '-': 27, '́': 2, ';': 1, '[': 1, '/': 1})

Train French Punctuation Count: Counter({'.': 5825, ',': 1774, "'": 1558, '’': 1378, '-': 1331, '?': 1007, '!': 645, '…': 301, '́': 75, '̀': 42, '̂': 31, ':': 21, '—': 20, '«': 12, ';': 11, '»': 11, '/': 9, '(': 7, ')': 7, '=': 6, '̧': 5, '"': 3, '“': 3, '”': 3, '–': 2, '&': 1, '€': 1})

Validate Dyula Punctuation Count: Counter({'.': 417, "'": 187, '?': 84, ',': 49, '!': 12, '-': 11, '[': 1})

Validate French Punctuation Count: Counter({'.': 1083, ',': 299, "'": 279, '-': 270, '’': 247, '?': 153, '!': 123, '…': 54, '́': 18, '̀': 11, '—': 8, ':': 5, '«': 4, '̂': 4, '»': 4, ';': 3, '̧': 1})

Test Dyula Punctuation Count: Counter({'.': 406, "'": 173, '?': 72, ',': 43, '!': 7, '-': 3, '&': 1})


In [248]:
test.head()

,ID,translation,dyu,dyu_punctuation
0,ID_17345911362699,"{'dyu': 'An kelen duron le tun be yi', 'fr': '0'}",An kelen duron le tun be yi,[]
1,ID_173626847.3381,"{'dyu': 'O ka papiye farana.', 'fr': '0'}",O ka papiye farana.,[.]
2,ID_17704632382547,"{'dyu': 'N tɔrɔla kɔ tuguni', 'fr': '0'}",N tɔrɔla kɔ tuguni,[]
3,ID_19793499384156,"{'dyu': 'I tun b'a daminɛ tan kɛ.', 'fr': '0'}",I tun b'a daminɛ tan kɛ.,"[', .]"
4,ID_17802727385575,"{'dyu': 'A kɛra ka ban.', 'fr': '0'}",A kɛra ka ban.,[.]


In [249]:
train = train.drop(['dyu_punctuation', 'fr_punctuation', 'translation'], axis=1)
test = test.drop(['dyu_punctuation', 'translation'], axis=1)
validate = validate.drop(['dyu_punctuation', 'fr_punctuation', 'translation'], axis=1)

In [250]:
train.head()

,ID,dyu,fr
0,ID_18897661270129,A bi ji min na,Il boit de l’eau.
1,ID_18479132727846,A le dalakolontɛ lon bɛ.,Il se plaint toujours.
2,ID_18164131280307,Mun? Fɛn dɔ.,Quoi ? Quelque chose.
3,ID_18344573728152,O bɛ bi bɔra fo Gubeta.,Tous sortent excepté Gubetta.
4,ID_18127342282717,A ale lo bi da bugɔ la!,Ah ! c’est lui… il sonne…


In [251]:
test.head()

,ID,dyu
0,ID_17345911362699,An kelen duron le tun be yi
1,ID_173626847.3381,O ka papiye farana.
2,ID_17704632382547,N tɔrɔla kɔ tuguni
3,ID_19793499384156,I tun b'a daminɛ tan kɛ.
4,ID_17802727385575,A kɛra ka ban.


In [252]:
validate.head()

,ID,dyu,fr
0,ID_17914990255818,I tɔgɔ bi cogodɔ,Tu portes un nom de fantaisie.
1,ID_18135961264225,Puɛn saba fɔlɔ.,Trois points d’avance.
2,ID_18161475265686,Tile bena,Le soleil s’est couché.
3,ID_17305345266745,cogoya kelen,Mêmes mouvements.
4,ID_18106593267767,N ma daraka dun ban.,Je n’ai pas encore déjeuné.


## Creating Sentence Corpus

In [253]:
train.head()

,ID,dyu,fr
0,ID_18897661270129,A bi ji min na,Il boit de l’eau.
1,ID_18479132727846,A le dalakolontɛ lon bɛ.,Il se plaint toujours.
2,ID_18164131280307,Mun? Fɛn dɔ.,Quoi ? Quelque chose.
3,ID_18344573728152,O bɛ bi bɔra fo Gubeta.,Tous sortent excepté Gubetta.
4,ID_18127342282717,A ale lo bi da bugɔ la!,Ah ! c’est lui… il sonne…


In [254]:
train_dyu_corpus = train['dyu'].tolist()
train_fr_corpus = train['fr'].tolist()

validate_dyu_corpus = validate['dyu'].tolist()
validate_fr_corpus = validate['fr'].tolist()

test_dyu_corpus = test['dyu'].tolist()

## Tokenization of the Sentences in Words

In [255]:
train_dyu_unique_words = set(word for sentence in train_dyu_corpus for word in sentence.split())
train_fr_unique_words = set(word for sentence in train_fr_corpus for word in sentence.split())

validate_dyu_unique_words = set(word for sentence in validate_dyu_corpus for word in sentence.split())
validate_fr_unique_words = set(word for sentence in validate_fr_corpus for word in sentence.split())

test_dyu_unique_words = set(word for sentence in test_dyu_corpus for word in sentence.split())

In [256]:
num_unique_dyu_words = len(train_dyu_unique_words)
num_unique_fr_words = len(train_fr_unique_words)

In [257]:
num_unique_dyu_words

9226

In [258]:
print('Total No. of Unique Words in dyu Train Vocab: ', num_unique_dyu_words)
print('Total No. of Unique Words in fr Train Vocab: ', num_unique_fr_words)

Total No. of Unique Words in dyu Train Vocab:  9226
Total No. of Unique Words in fr Train Vocab:  12619


## Converting Words into numbers

In [259]:
dyu_corpus = train['dyu'].tolist() + validate['dyu'].tolist() + test['dyu'].tolist()
fr_corpus = train['fr'].tolist() + validate['fr'].tolist()

In [260]:
dyu_tokenizer = Tokenizer()
dyu_tokenizer.fit_on_texts(dyu_corpus)

fr_tokenizer = Tokenizer()
fr_tokenizer.fit_on_texts(fr_corpus)

In [261]:
train_dyu_sequences = dyu_tokenizer.texts_to_sequences(train['dyu'].tolist())
train_fr_sequences = fr_tokenizer.texts_to_sequences(train['fr'].tolist())

validate_dyu_sequences = dyu_tokenizer.texts_to_sequences(validate['dyu'].tolist())
validate_fr_sequences = fr_tokenizer.texts_to_sequences(validate['fr'].tolist())

In [262]:
test_dyu_sequences = dyu_tokenizer.texts_to_sequences(test['dyu'].tolist())

In [263]:
print("First 5 Dyula sequences in training data:", train_dyu_sequences[:5])
print("First 5 French sequences in training data:", train_fr_sequences[:5])

First 5 Dyula sequences in training data: [[1, 3, 171, 22, 7], [1, 9, 2882, 62, 33], [121, 72, 39], [10, 33, 3, 113, 97, 1786], [1, 47, 14, 3, 89, 355, 12]]
First 5 French sequences in training data: [[3, 1564, 1, 419], [3, 27, 1190, 169], [153, 240, 129], [88, 827, 2263, 2264], [74, 24, 1565, 3, 2265]]


### Adding Padding to make Each Sentence of the same Length

In [264]:
max_length_dyu = max(len(seq) for seq in dyu_tokenizer.texts_to_sequences(dyu_corpus))
max_length_fr = max(len(seq) for seq in fr_tokenizer.texts_to_sequences(fr_corpus))

print(f"Maximum length of a sentence in Dyula corpus: {max_length_dyu}")
print(f"Maximum length of a sentence in French corpus: {max_length_fr}")

Maximum length of a sentence in Dyula corpus: 30
Maximum length of a sentence in French corpus: 19


In [265]:
train_dyu_padded = pad_sequences(train_dyu_sequences, padding='post')
train_fr_padded = pad_sequences(train_fr_sequences, padding='post')

validate_dyu_padded = pad_sequences(validate_dyu_sequences, padding='post')
validate_fr_padded = pad_sequences(validate_fr_sequences, padding='post')

test_dyu_padded = pad_sequences(test_dyu_sequences, padding='post')

In [266]:
print("First 2 padded Dyula sequences in training data:", train_dyu_padded[:2])
print("First 2 padded French sequences in training data:", train_fr_padded[:2])

First 2 padded Dyula sequences in training data: [[   1    3  171   22    7    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0]
 [   1    9 2882   62   33    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0]]
First 2 padded French sequences in training data: [[   3 1564    1  419    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0]
 [   3   27 1190  169    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0]]


## Preprocess Pipeline

In [267]:
def tokenize(sentences):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(sentences)
    sequences = tokenizer.texts_to_sequences(sentences)
    return sequences, tokenizer

In [268]:

def pad(sequences, maxlen=None):
    return pad_sequences(sequences, padding='post', maxlen=maxlen)

In [269]:
def preprocess(x, y):
    preprocess_x, x_tk = tokenize(x)
    preprocess_y, y_tk = tokenize(y)

    preprocess_x = pad(preprocess_x)
    preprocess_y = pad(preprocess_y)

    preprocess_y = preprocess_y.reshape(*preprocess_y.shape, 1)

    return preprocess_x, preprocess_y, x_tk, y_tk

In [270]:
test.head()

,ID,dyu
0,ID_17345911362699,An kelen duron le tun be yi
1,ID_173626847.3381,O ka papiye farana.
2,ID_17704632382547,N tɔrɔla kɔ tuguni
3,ID_19793499384156,I tun b'a daminɛ tan kɛ.
4,ID_17802727385575,A kɛra ka ban.


In [271]:
dyu_sentences = train['dyu'].tolist() + validate['dyu'].tolist()
fr_sentences = train['fr'].tolist() + validate['fr'].tolist()

In [272]:
preproc_dyu_sentences, preproc_fr_sentences, dyu_tokenizer, fr_tokenizer = preprocess(dyu_sentences, fr_sentences)

In [273]:
np.set_printoptions(threshold=np.inf)

In [274]:
dyu_tokenizer

In [275]:
train_size = len(train)
validate_size = len(validate)
test_size = len(test)

In [276]:
train_preproc_dyu = preproc_dyu_sentences[:train_size]
validate_preproc_dyu = preproc_dyu_sentences[train_size:train_size + validate_size]
test_preproc_dyu = preproc_dyu_sentences[train_size + validate_size:]

In [277]:
train_preproc_fr = preproc_fr_sentences[:train_size]
validate_preproc_fr = preproc_fr_sentences[train_size:train_size + validate_size]
test_preproc_fr = preproc_fr_sentences[train_size + validate_size:]

In [278]:
train_preproc_fr = train_preproc_fr.reshape(train_preproc_fr.shape[0], train_preproc_fr.shape[1])
validate_preproc_fr = validate_preproc_fr.reshape(validate_preproc_fr.shape[0], validate_preproc_fr.shape[1])
test_preproc_fr = test_preproc_fr.reshape(test_preproc_fr.shape[0], test_preproc_fr.shape[1])


In [279]:
train['dyu_preprocessed'] = list(train_preproc_dyu)
train['fr_preprocessed'] = list(train_preproc_fr)

validate['dyu_preprocessed'] = list(validate_preproc_dyu)
validate['fr_preprocessed'] = list(validate_preproc_fr)

In [280]:
train.head()

,ID,dyu,fr,dyu_preprocessed,fr_preprocessed
0,ID_18897661270129,A bi ji min na,Il boit de l’eau.,"[1, 3, 169, 22, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[3, 1564, 1, 419, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,ID_18479132727846,A le dalakolontɛ lon bɛ.,Il se plaint toujours.,"[1, 9, 2597, 66, 32, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[3, 27, 1190, 169, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,ID_18164131280307,Mun? Fɛn dɔ.,Quoi ? Quelque chose.,"[124, 71, 39, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[153, 240, 129, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,ID_18344573728152,O bɛ bi bɔra fo Gubeta.,Tous sortent excepté Gubetta.,"[10, 32, 3, 121, 102, 1606, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[88, 827, 2263, 2264, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,ID_18127342282717,A ale lo bi da bugɔ la!,Ah ! c’est lui… il sonne…,"[1, 47, 14, 3, 93, 345, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[74, 24, 1565, 3, 2265, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


In [281]:
validate.head()

,ID,dyu,fr,dyu_preprocessed,fr_preprocessed
0,ID_17914990255818,I tɔgɔ bi cogodɔ,Tu portes un nom de fantaisie.,"[16, 81, 3, 7818, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[23, 2174, 12, 152, 1, 3554, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,ID_18135961264225,Puɛn saba fɔlɔ.,Trois points d’avance.,"[1613, 78, 142, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[48, 992, 2278, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,ID_18161475265686,Tile bena,Le soleil s’est couché.,"[368, 286, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[4, 885, 541, 1702, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,ID_17305345266745,cogoya kelen,Mêmes mouvements.,"[645, 34, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[570, 1119, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,ID_18106593267767,N ma daraka dun ban.,Je n’ai pas encore déjeuné.,"[6, 18, 1521, 1019, 126, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[8, 284, 11, 124, 2142, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


In [282]:
test.head()

,ID,dyu
0,ID_17345911362699,An kelen duron le tun be yi
1,ID_173626847.3381,O ka papiye farana.
2,ID_17704632382547,N tɔrɔla kɔ tuguni
3,ID_19793499384156,I tun b'a daminɛ tan kɛ.
4,ID_17802727385575,A kɛra ka ban.


## Word Ids back to Text

In [283]:
def logits_to_text(logits, tokenizer):

    index_to_words = {id: word for word, id in tokenizer.word_index.items()}
    index_to_words[0] = '<PAD>'

    return ' '.join([index_to_words[np.argmax(prediction)] for prediction in logits])


In [284]:
## Testing the funciton logits_to_text

sample_logits = np.array([
    [0.1, 0.3, 0.6],
    [0.2, 0.4, 0.4],
    [0.5, 0.2, 0.3]
])

translated_text = logits_to_text(sample_logits, fr_tokenizer)
print(translated_text)

la de <PAD>


## Model 1: RNN (Many - to - Many)

In [285]:
max_dyu_sequence_length = max(len(seq) for seq in preproc_dyu_sentences)
max_fr_sequence_length = max(len(seq) for seq in preproc_fr_sentences)
dyu_vocab_size = len(dyu_tokenizer.word_index) + 1  # +1 for padding token
fr_vocab_size = len(fr_tokenizer.word_index) + 1  # +1 for padding token

In [286]:
from keras.models import Sequential
from keras.layers import GRU, Dense, TimeDistributed, Dropout
from keras.optimizers import Adam
from keras.losses import sparse_categorical_crossentropy


In [287]:
def simple_model(input_shape, output_sequence_length, dyu_vocab_size, fr_vocab_size):
    learning_rate = 0.005
    
    model = Sequential()
    model.add(GRU(256, input_shape=input_shape[1:], return_sequences=True))
    model.add(TimeDistributed(Dense(1024, activation='relu')))
    model.add(Dropout(0.5))
    model.add(TimeDistributed(Dense(fr_vocab_size, activation='softmax')))
    
    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate),
                  metrics=['accuracy'])
    return model

In [291]:
tmp_x = pad_sequences(preproc_dyu_sentences, maxlen=max_fr_sequence_length, padding='post')
tmp_x = tmp_x.reshape((-1, max_fr_sequence_length, 1))

In [289]:
# tmp_x = pad_sequences(preproc_dyu_sentences, maxlen=max_fr_sequence_length, padding='post')
# tmp_x = tmp_x.reshape((-1, max_fr_sequence_length))


In [292]:
simple_rnn_model = simple_model(
    tmp_x.shape,
    max_fr_sequence_length,
    dyu_vocab_size,
    fr_vocab_size)

In [293]:

print(simple_rnn_model.summary())

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_5 (GRU)                     │ (None, 19, 256)        │       198,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 19, 1024)       │       263,168 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 19, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 19, 11052)      │    11,328,300 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,790,380 (44.98 MB)

 Trainable params: 11,790,380 (44.98 MB)

 Non-trainable params: 0 (0.00 B)

None


In [294]:
simple_rnn_model.fit(tmp_x, np.array(preproc_fr_sentences), batch_size=100, epochs=1)

10/10 ━━━━━━━━━━━━━━━━━━━━ 344s 32s/step - accuracy: 0.6862 - loss: 7.4175


In [297]:
import numpy as np

def preprocess_single_sentence(sentence, tokenizer, max_sequence_length):
    sequence = tokenizer.texts_to_sequences([sentence])
    padded_sequence = pad_sequences(sequence, maxlen=max_sequence_length, padding='post')
    return padded_sequence.reshape((1, max_sequence_length, 1))

def logits_to_text(logits, tokenizer):
    index_to_words = {id: word for word, id in tokenizer.word_index.items()}
    index_to_words[0] = '<PAD>'
    return ' '.join([index_to_words[np.argmax(prediction)] for prediction in logits])

dyu_sentence = validate['dyu'].iloc[2]
actual_fr_sentence = validate['fr'].iloc[2]

preproc_dyu_sentence = preprocess_single_sentence(dyu_sentence, dyu_tokenizer, max_fr_sequence_length)

predicted_logits = simple_rnn_model.predict(preproc_dyu_sentence)

predicted_fr_sentence = logits_to_text(predicted_logits[0], fr_tokenizer)

# Print the results
print("Dyula Sentence:", dyu_sentence)
print("Actual French Sentence:", actual_fr_sentence)
print("Predicted French Sentence:", predicted_fr_sentence)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 265ms/step
Dyula Sentence: Tile bena
Actual French Sentence: Le soleil s’est couché.
Predicted French Sentence: il il la la du <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>


In [ ]:
print("Predicted French sentence:")
print(logits_to_text(simple_rnn_model.predict(tmp_x[:1])[0], fr_tokenizer))

print("\nCorrect Translation:")
print(fr_sentences[:1])

print("\nOriginal text:")
print(dyu_sentences[:1])

Predicted French sentence:
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
<PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>

Correct Translation:
['Il boit de l’eau.']

Original text:
['A bi ji min na']
